In [1]:
import pandas as pd

df = pd.read_csv("clean_stock_data.csv")

df.head()

,trade_id,ticker,trade_date,open_price,high_price,low_price,close_price,volume,market_cap_usd,pe_ratio,dividend_yield,beta,sector,portfolio,currency,analyst_rating,shares_held,purchase_price
0,Trd00742,META,01-01-2020,632.80,662.89,618.75,648.67,29574983,1.617168e+09,49.13,1.816,0.421,Financial,Portfolio_B,Usd,Buy,2387,576.30
1,Trd01102,JPM,02-01-2020,735.46,739.73,652.32,682.63,38337831,3.740649e+09,57.46,2.387,1.801,Technology,Portfolio_A,Usd,Sell,828,631.73
2,Trd00595,AMZN,03-01-2020,659.44,660.69,604.69,607.69,10089375,5.539461e+09,20.57,2.486,0.628,Consumer,Portfolio_B,Eur,Sell,2729,659.13
3,Trd00686,GOOGL,03-01-2020,522.12,571.82,502.52,555.74,23904732,3.563607e+09,7.04,3.578,2.348,Technology,Portfolio_C,Usd,Buy,1923,534.67
4,Trd01812,META,03-01-2020,642.01,676.82,622.18,659.04,25562169,1.949395e+09,79.19,2.235,1.915,Financial,Portfolio_A,Eur,Buy,1443,628.63


In [2]:
# Check the columns
print(df.columns)

Index(['trade_id', 'ticker', 'trade_date', 'open_price', 'high_price',
       'low_price', 'close_price', 'volume', 'market_cap_usd', 'pe_ratio',
       'dividend_yield', 'beta', 'sector', 'portfolio', 'currency',
       'analyst_rating', 'shares_held', 'purchase_price'],
      dtype='str')


In [10]:
# Convert mixed date formats
df["trade_date"] = pd.to_datetime(
    df["trade_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

In [4]:
df = df.sort_values(["ticker", "trade_date"])

In [5]:
# Calculate Daily Return
df["daily_return"] = (
    df.groupby("ticker")["close_price"]
      .pct_change()
)

In [6]:
df["daily_return_pct"] = df["daily_return"] * 100

In [12]:
# Sort data
df = df.sort_values(["ticker", "trade_date"])

# Calculate daily return
df["daily_return"] = df.groupby("ticker")["close_price"].pct_change()

# Remove infinite values using Pandas
df["daily_return"] = df["daily_return"].replace(
    [float("inf"), float("-inf")], None
)

# Calculate cumulative return
df["cumulative_return"] = (
    df.groupby("ticker")["daily_return"]
      .transform(lambda x: (1 + x).cumprod() - 1)
)

In [8]:
# Calculate CAGR
cagr_data = (
    df.groupby("ticker")
      .agg(
          beginning_price=("close_price", "first"),
          ending_price=("close_price", "last"),
          beginning_date=("trade_date", "first"),
          ending_date=("trade_date", "last")
      )
      .reset_index()
)

In [13]:
# Convert dates to datetime
cagr_data["beginning_date"] = pd.to_datetime(
    cagr_data["beginning_date"],
    format="mixed",
    errors="coerce"
)

cagr_data["ending_date"] = pd.to_datetime(
    cagr_data["ending_date"],
    format="mixed",
    errors="coerce"
)

# Calculate number of years
cagr_data["years"] = (
    cagr_data["ending_date"] - cagr_data["beginning_date"]
).dt.days / 365.25

# Check result
print(cagr_data[[
    "beginning_date",
    "ending_date",
    "years"
]].head())

  beginning_date ending_date     years
0     2021-01-02  2023-03-31  2.239562
1     2023-01-02  2024-01-31  1.078713
2     2024-01-02  2022-10-31 -1.171800
3     2024-01-02  2020-11-30 -3.088296
4     2022-01-06  2021-12-31 -0.016427


In [15]:
cagr_data = (
    df.groupby("ticker")
      .agg(
          beginning_price=("close_price", "first"),
          ending_price=("close_price", "last"),
          beginning_date=("trade_date", "first"),
          ending_date=("trade_date", "last")
      )
      .reset_index()
)

In [17]:
cagr_data["years"] = (
    cagr_data["ending_date"] - cagr_data["beginning_date"]
).dt.days / 365.25

In [18]:
cagr_data["cagr"] = (
    (cagr_data["ending_price"] / cagr_data["beginning_price"])
    ** (1 / cagr_data["years"]) - 1
)

In [19]:
cagr_data["cagr_pct"] = cagr_data["cagr"] * 100

In [20]:
cagr_data[
    ["ticker", "beginning_price", "ending_price", "years", "cagr_pct"]
]

,ticker,beginning_price,ending_price,years,cagr_pct
0,AAPL,353.100,460.84,4.791239,5.715415
1,AMD,397.050,574.77,4.963723,7.736929
2,AMZN,607.690,146.68,4.941821,-24.996029
3,BAC,472.660,821.33,4.906229,11.920865
4,BRK,159.940,572.32,4.950034,29.376126
5,GOOGL,555.740,392.89,4.941821,-6.776530
6,GS,515.540,350.40,4.941821,-7.516238
7,INTC,350.120,357.21,4.939083,0.406728
8,JPM,682.630,501.68,4.884326,-6.110997
9,META,648.670,444.51,4.996578,-7.285201


In [25]:
# Calculate Unrealised P&L
df["price_change"] = (
    df["close_price"] - df["open_price"]
)

In [28]:
# Calculate daily return
df["daily_return"] = (
    df.groupby("ticker")["close_price"]
      .pct_change()
)

# Replace invalid values
df["daily_return"] = df["daily_return"].replace(
    [float("inf"), float("-inf")], None
)

# Calculate cumulative return
df["cumulative_return"] = (
    df.groupby("ticker")["daily_return"]
      .transform(lambda x: (1 + x.fillna(0)).cumprod() - 1)
)

# Convert to percentage
df["daily_return_pct"] = df["daily_return"] * 100
df["cumulative_return_pct"] = df["cumulative_return"] * 100

In [33]:
# Calculate Unrealised P&L per share
df["unrealised_pnl"] = (
    df["close_price"] - df["open_price"]
)

In [34]:
df[
    [
        "ticker",
        "trade_date",
        "close_price",
        "daily_return_pct",
        "cumulative_return_pct",
        "unrealised_pnl"
    ]
].head(10)

,ticker,trade_date,close_price,daily_return_pct,cumulative_return_pct,unrealised_pnl
58,AAPL,2020-01-30,353.10,NaN,0,1.75
87,AAPL,2020-02-28,316.25,-10.436137,-10.436137,-18.69
122,AAPL,2020-03-31,283.00,-10.513834,-19.852733,-14.31
156,AAPL,2020-04-24,237.86,-15.95053,-32.636647,-14.56
166,AAPL,2020-05-05,746.97,214.037669,111.546304,13.49
189,AAPL,2020-06-01,473.32,-36.634671,34.047012,31.62
190,AAPL,2020-06-02,650.29,37.389081,84.165959,-2.66
201,AAPL,2020-06-17,463.72,-28.690277,31.328236,15.98
251,AAPL,2020-07-28,288.63,-37.757699,-18.258284,-0.14
252,AAPL,2020-07-28,656.75,127.54045,85.995469,-19.51


In [35]:
df.to_csv("trading_returns_analysis.csv", index=False)